# Sentence splitting

Sentence splitting using SpaCy:

In [36]:
import spacy
import pandas as pd

reviews = pd.read_csv("training_set/Reviews.csv", encoding="cp1252")
test = pd.read_csv("test_sets/Sentiment-topic-test.tsv", sep="\t")

nlp = spacy.load("en_core_web_sm")

def split_into_sentences(df, text_col="ReviewContent"):
    sentence_rows = []
    for review_id, text in df[text_col].dropna().items():
        doc = nlp(text)
        for sentence_id, sent in enumerate(doc.sents):
            sentence_text = sent.text.strip()

            if sentence_text:
                sentence_rows.append({
                    "review_id": review_id,
                    "sentence_id": sentence_id,
                    "sentence": sentence_text

                })
    return pd.DataFrame(sentence_rows)


sentences_df = split_into_sentences(reviews, text_col="ReviewContent")
for _, row in sentences_df.head(20).iterrows():
    print(f"Review {row['review_id']} | Sentence {row['sentence_id']}: {row['sentence']}")


Review 0 | Sentence 0: Good.
Review 0 | Sentence 1: It IS a page turner.
Review 0 | Sentence 2: You can read this book in one day, two at the most, and the plot drives the whole book.
Review 0 | Sentence 3: The unreliable narrators (there are two besides the main character) are as unlikable as they are unreliable, and there isn't a nice male in the book.
Review 0 | Sentence 4: Entirely plot driven; the characters are paper thin.
Review 0 | Sentence 5: You can figure out who-dunnit by the middle of the book.
Review 0 | Sentence 6: The ending is weak.
Review 0 | Sentence 7: I can't imagine what all the fuss is about, except that it is quick and there are lots of twists and turns, and you can't trust anyone to tell the truth.
Review 1 | Sentence 0: There are no words for how much I loathed this book.
Review 1 | Sentence 1: This was the first audiobook I ever listened to so not sure if something was lost in translation here, if I just disliked the narrators, or if it is truly the book I ha

# VADER sentiment - 1st system

In [37]:
import nltk
from nltk.sentiment import vader
from nltk.sentiment.vader import SentimentIntensityAnalyzer
vader_model = SentimentIntensityAnalyzer()
sentiment_scores = []

def vader_sentiment(sentence):
    score = vader_model.polarity_scores(sentence)["compound"]

    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

for r_id, s_id, sent in sentences_df[["review_id", "sentence_id", "sentence"]].itertuples(index=False):
    sentiment_scores.append({

                    "review_id": r_id,
                    "sentence_id": s_id,
                    "sentence": sent,
                    "score": vader_model.polarity_scores(sent),
                    "vader_label": vader_sentiment(sent)
                })
    
vader_df = pd.DataFrame(sentiment_scores)

for _, row in vader_df.head(20).iterrows():
    print(f"Review {row['review_id']} | Sentence {row['sentence_id']}: {row['sentence']}")
    print(f"Score: {row['score']}")
    print(f"Label: {row['vader_label']}\n")


Review 0 | Sentence 0: Good.
Score: {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.4404}
Label: positive

Review 0 | Sentence 1: It IS a page turner.
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label: neutral

Review 0 | Sentence 2: You can read this book in one day, two at the most, and the plot drives the whole book.
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label: neutral

Review 0 | Sentence 3: The unreliable narrators (there are two besides the main character) are as unlikable as they are unreliable, and there isn't a nice male in the book.
Score: {'neg': 0.089, 'neu': 0.911, 'pos': 0.0, 'compound': -0.3252}
Label: negative

Review 0 | Sentence 4: Entirely plot driven; the characters are paper thin.
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label: neutral

Review 0 | Sentence 5: You can figure out who-dunnit by the middle of the book.
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label: neutral

Review 0 | S

# Text blob: sentiment analysis, system 2

In [38]:
from textblob import TextBlob

def textblob_sentiment(sentence):

    polarity = TextBlob(sentence).sentiment.polarity

    if polarity > 0:
        return "positive"
    elif polarity < 0:
        return "negative"
    else:
        return "neutral"

textblob_scores = []

for r_id, s_id, sent in sentences_df[["review_id", "sentence_id", "sentence"]].itertuples(index=False):

    blob = TextBlob(sent)
    textblob_scores.append({

        "review_id": r_id,
        "sentence_id": s_id,
        "sentence": sent,
        "textblob_polarity": blob.sentiment.polarity,
        "textblob_subjectivity": blob.sentiment.subjectivity,
        "textblob_label": textblob_sentiment(sent)

    })
textblob_df = pd.DataFrame(textblob_scores)
textblob_df.head(20)

,review_id,sentence_id,sentence,textblob_polarity,textblob_subjectivity,textblob_label
0,0,0,Good.,0.700000,0.600000,positive
1,0,1,It IS a page turner.,0.000000,0.000000,neutral
2,0,2,"You can read this book in one day, two at the ...",0.350000,0.450000,positive
3,0,3,The unreliable narrators (there are two beside...,0.255556,0.477778,positive
4,0,4,Entirely plot driven; the characters are paper...,-0.200000,0.737500,negative
5,0,5,You can figure out who-dunnit by the middle of...,0.000000,0.000000,neutral
6,0,6,The ending is weak.,-0.375000,0.625000,negative
7,0,7,"I can't imagine what all the fuss is about, ex...",0.333333,0.500000,positive
8,1,0,There are no words for how much I loathed this...,0.200000,0.200000,positive
9,1,1,This was the first audiobook I ever listened t...,-0.275000,0.630556,negative


In [ ]:
comparison_df = vader_df.merge(
    textblob_df,
    on=["review_id", "sentence_id", "sentence"]
)
disagreements = comparison_df[
    comparison_df["vader_label"] != comparison_df["textblob_label"]
]
disagreements.head(20)

,review_id,sentence_id,sentence,score,vader_label,textblob_polarity,textblob_subjectivity,textblob_label
2,0,2,"You can read this book in one day, two at the ...","{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",neutral,0.350000,0.450000,positive
3,0,3,The unreliable narrators (there are two beside...,"{'neg': 0.089, 'neu': 0.911, 'pos': 0.0, 'comp...",negative,0.255556,0.477778,positive
4,0,4,Entirely plot driven; the characters are paper...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",neutral,-0.200000,0.737500,negative
7,0,7,"I can't imagine what all the fuss is about, ex...","{'neg': 0.082, 'neu': 0.848, 'pos': 0.07, 'com...",negative,0.333333,0.500000,positive
8,1,0,There are no words for how much I loathed this...,"{'neg': 0.398, 'neu': 0.602, 'pos': 0.0, 'comp...",negative,0.200000,0.200000,positive
15,1,7,"I poured myself another drink, I don't want t...","{'neg': 0.149, 'neu': 0.851, 'pos': 0.0, 'comp...",negative,0.000000,0.000000,neutral
16,1,8,I can't stop myself.,"{'neg': 0.0, 'neu': 0.514, 'pos': 0.486, 'comp...",positive,0.000000,0.000000,neutral
18,1,10,Not a direct quote from the book but you get t...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",neutral,-0.050000,0.400000,negative
21,2,0,I think I would ordinarily cut this book more ...,"{'neg': 0.075, 'neu': 0.885, 'pos': 0.04, 'com...",negative,0.088272,0.443210,positive
22,2,1,But because the book has received so many rave...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",neutral,0.266667,0.566667,positive


# NERC 

Function extracting the entities:

In [42]:
# Build BIO tags at token level (entity-only BIO: B-ENT / I-ENT / O)
# NOTE: Reviews.csv has no manual NER labels, so these are weak labels from spaCy.

def sentence_to_bio_rows(sentence_text, sentence_id):
    doc = nlp(sentence_text)

    bio_rows = []
    token_id = 0

    for token in doc:
        if token.is_space:
            continue

        if token.ent_iob_ == "B":
            bio_tag = "B-ENT"
        elif token.ent_iob_ == "I":
            bio_tag = "I-ENT"
        else:
            bio_tag = "O"

        bio_rows.append({
            "sentence id": sentence_id,
            "token id": token_id,
            "token": token.text,
            "BIO NER tag": bio_tag
        })
        token_id += 1

    return bio_rows

Apply NERC to all sentences:

In [43]:
# Apply sentence-level BIO tagging to split review sentences
bio_rows = []
global_sentence_id = 0

for sentence_text in sentences_df["sentence"]:
    bio_rows.extend(sentence_to_bio_rows(sentence_text, global_sentence_id))
    global_sentence_id += 1

ner_bio_df = pd.DataFrame(bio_rows)
#ner_bio_df.head(30)


In [34]:
print(ner_bio_df.sample(20, random_state=42))

        sentence id  token id       token BIO NER tag
191193        10856        19           .           O
199341        11323         7      though           O
405138        23730        15  characters           O
472904        28280         2           a           O
364216        21131        10        make           O
42835          2373         5      turned           O
171884         9743        15        need           O
371228        21581        17         pay           O
233602        13212        10        told           O
327519        18855         0       Great           O
27571          1480         3     mystery           O
328610        18926        12          of           O
212770        12088         1       loved           O
47522          2581        31      liking           O
2253            102         6           -           O
74685          4151         4   surprised           O
330839        19066        20           .           O
438793        25966         

# NER on the test set

Run NER BIO on test sentences:

In [ ]:
test_bio_rows = []

# sentence-level BIO tagging to test set sentences
for _, row in test[["sentence id", "text"]].iterrows():
    sent_id = int(row["sentence id"])
    sent_text = row["text"]
    test_bio_rows.extend(sentence_to_bio_rows(sent_text, sent_id))

# DataFrame for the test set BIO tags
test_ner_df = pd.DataFrame(test_bio_rows)
test_ner_df.head(30)

,sentence id,token id,token,BIO NER tag
0,0,0,It,O
1,0,1,took,O
2,0,2,eight,B-ENT
3,0,3,years,I-ENT
4,0,4,for,O
5,0,5,Warner,B-ENT
6,0,6,Brothers,I-ENT
7,0,7,to,O
8,0,8,recover,O
9,0,9,from,O


The predictions:

In [50]:
test_ner_df = test_ner_df[["sentence id", "token id", "token", "BIO NER tag"]]
test_ner_df.to_csv("outputs/ner_test_predictions.tsv", sep="\t", index=False)

print(test_ner_df["BIO NER tag"].value_counts())

BIO NER tag
O        178
B-ENT     20
I-ENT     16
Name: count, dtype: int64


# Evaluate NER accuracy: 

In [53]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

gold_ner = pd.read_csv("test_sets/NER-test.tsv", sep="\t")

test_ner_eval = gold_ner.merge(
    test_ner_df,  # use your existing variable directly
    on=["sentence id", "token id", "token"],
    how="left",
    suffixes=("_gold", "_pred")
)

test_ner_eval["BIO NER tag_pred"] = test_ner_eval["BIO NER tag_pred"].fillna("O")

def to_bio_only(tag):
    if isinstance(tag, str):
        if tag.startswith("B-"):
            return "B"
        if tag.startswith("I-"):
            return "I"
    return "O"

test_ner_eval["bio_gold"] = test_ner_eval["BIO NER tag_gold"].apply(to_bio_only)
test_ner_eval["bio_pred"] = test_ner_eval["BIO NER tag_pred"].apply(to_bio_only)

print("NER BIO accuracy:", accuracy_score(test_ner_eval["bio_gold"], test_ner_eval["bio_pred"]))
print(classification_report(test_ner_eval["bio_gold"], test_ner_eval["bio_pred"]))

labels = ["B", "I", "O"]
ner_confusion_matrix = pd.DataFrame(
    confusion_matrix(test_ner_eval["bio_gold"], test_ner_eval["bio_pred"], labels=labels),
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels]
)
ner_confusion_matrix

NER BIO accuracy: 0.9439252336448598
              precision    recall  f1-score   support

           B       0.70      0.82      0.76        17
           I       0.75      0.86      0.80        14
           O       0.99      0.96      0.98       183

    accuracy                           0.94       214
   macro avg       0.81      0.88      0.84       214
weighted avg       0.95      0.94      0.95       214



,pred_B,pred_I,pred_O
true_B,14,1,2
true_I,2,12,0
true_O,4,3,176


In [16]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Topic analysis 

In [29]:
# the possible topics
topic_profiles = {
    "book": "book novel author writer reading pages chapter plot story character literature ending prose",
    "movie": "movie film cinema actor actress director scene screen role cast trailer script",
    "restaurant": "restaurant food menu dish service table dinner lunch meal waiter chef atmosphere"
}

topic_labels = list(topic_profiles.keys())

# TF-IDF vectorizer 
topic_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

# the topic profile matrix
topic_profile_matrix = topic_vectorizer.fit_transform(topic_profiles.values())

Topic prediction function:

In [30]:
# function to predict the topic of a sentence
def predict_topic(sentence):
    sentence_matrix = topic_vectorizer.transform([sentence])
    similarities = cosine_similarity(sentence_matrix, topic_profile_matrix)

    best_topic_index = similarities.argmax()
    predicted_topic = topic_labels[best_topic_index]
    confidence_score = similarities.max()

    return predicted_topic, confidence_score

Topic analysis per sentence:

In [31]:
topic_predictions = sentences_df["sentence"].apply(predict_topic)

# add the predicted topic and confidence score
sentences_df["topic_label"] = topic_predictions.apply(lambda x: x[0])
sentences_df["topic_score"] = topic_predictions.apply(lambda x: x[1])

The first 20 topic results:

In [32]:
sentences_df[[
    "review_id",
    "sentence_id",
    "sentence",
    "topic_label",
    "topic_score"
]].head(20)

,review_id,sentence_id,sentence,topic_label,topic_score
0,0,0,Good.,book,0.000000
1,0,1,It IS a page turner.,book,0.000000
2,0,2,"You can read this book in one day, two at the ...",book,0.268328
3,0,3,The unreliable narrators (there are two beside...,book,0.282843
4,0,4,Entirely plot driven; the characters are paper...,book,0.200000
5,0,5,You can figure out who-dunnit by the middle of...,book,0.200000
6,0,6,The ending is weak.,book,0.200000
7,0,7,"I can't imagine what all the fuss is about, ex...",book,0.000000
8,1,0,There are no words for how much I loathed this...,book,0.200000
9,1,1,This was the first audiobook I ever listened t...,book,0.200000


# Topic analysis on the test set

In [46]:
# the distribution of topics in the test set
test["topic"].value_counts()
topic_predictions = test["text"].apply(predict_topic)

# add the predicted topic and confidence score to the test set
test["predicted_topic"] = topic_predictions.apply(lambda x: x[0])
test["topic_score"] = topic_predictions.apply(lambda x: x[1])

The results:

In [ ]:
test[[
    "sentence id",
    "text",
    "topic",
    "predicted_topic",
    "topic_score"
]]

,sentence id,text,topic,predicted_topic,topic_score
0,0,It took eight years for Warner Brothers to rec...,movie,movie,0.208514
1,1,All the New York University students love this...,restaurant,restaurant,0.208514
2,2,This Italian place is really trendy but they h...,restaurant,restaurant,0.361158
3,3,"In conclusion, my review of this book would be...",book,book,0.200000
4,4,The story of this movie is focused on Carl Bra...,movie,movie,0.147442
5,5,Chris O'Donnell stated that while filming for ...,movie,movie,0.208514
6,6,My husband and I moved to Amsterdam 6 years ag...,restaurant,book,0.000000
7,7,Dame Maggie Smith performed her role excellent...,movie,movie,0.208514
8,8,The new movie by Mr. Kruno was shot in New Yor...,movie,movie,0.147442
9,9,"I always have loved English novels, but I just...",book,book,0.000000


# Evaluate topic accuracy: 

In [ ]:
topic_accuracy = accuracy_score(
    test["topic"],
    test["predicted_topic"]
)

print("Topic accuracy:", topic_accuracy)

print(classification_report(
    test["topic"],
    test["predicted_topic"]
))

labels = ["book", "movie", "restaurant"]

topic_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        test["topic"],
        test["predicted_topic"],
        labels=labels
    ),
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

topic_confusion_matrix

Topic accuracy: 0.9
              precision    recall  f1-score   support

        book       0.67      1.00      0.80         2
       movie       1.00      1.00      1.00         5
  restaurant       1.00      0.67      0.80         3

    accuracy                           0.90        10
   macro avg       0.89      0.89      0.87        10
weighted avg       0.93      0.90      0.90        10



,pred_book,pred_movie,pred_restaurant
true_book,2,0,0
true_movie,0,5,0
true_restaurant,1,0,2
